# How much history should you use?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/how-much-history)

This notebook reproduces every figure and every number in the article.

## How to run it

| | |
| --- | --- |
| **Run it, nothing to install** | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/how-much-history.ipynb) |
| **Open it in VS Code** | [Open in VS Code](https://vscode.dev/github/Arthurnoo/out-of-sample/blob/main/notebooks/how-much-history.ipynb) |

Colab runs it in your browser. VS Code opens the real editor, also in your
browser, with nothing to install. Either way the next cell installs whatever is
missing, so you can simply **Run All**. The first run downloads prices and takes
a minute or two.

On your own machine instead: `git clone https://github.com/Arthurnoo/out-of-sample`

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

| Knob | What it does | Try |
| --- | --- | --- |
| `SYMBOL` | the market studied | `QQQ`, `GLD`, `BTC-USD` |
| `START` | how far back the data goes | `2000-01-01` for a shorter sample |
| `HORIZON` | how far ahead you forecast volatility | **the one that moves the answer most** |
| `WINDOWS` | window lengths compared | add `3`, add `10_000` |
| `LAMBDAS` | exponential decay rates tried | add `0.85`, `0.999` |
| `PAIR` | the two assets for the correlation test | `('SPY','GLD')` |

`HORIZON` is the interesting one. The optimal window is not a property of
volatility, it is a property of **the question you asked**. Forecast tomorrow
and the optimum shortens; forecast next year and it lengthens. Change it and
watch.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────
SYMBOL  = '^GSPC'
START   = '1950-01-01'
HORIZON = 21                # forecast realised volatility over the next N days

WINDOWS = [5, 10, 21, 42, 63, 126, 252, 504, 1260, 2520, 5040]
LAMBDAS = [0.80, 0.90, 0.94, 0.97, 0.99, 0.995]
PAIR    = ('SPY', 'TLT')
PAIR_START = '2000-01-01'

BARS_PER_YEAR = 252

In [ ]:
def get_returns(symbol, start):
    h = yf.Ticker(symbol).history(start=start, auto_adjust=True)
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    return h['Close'].pct_change().dropna()

r = get_returns(SYMBOL, START)
print(f"{len(r)} days, {r.index[0].date()} to {r.index[-1].date()}")

## 1. The criterion

The instinct is to pick the window whose estimate looks nicest. That is exactly
backwards: a long window always looks nicer, because it is smoother.

The right test is **how well the estimate predicts something it has not seen.**

The `.shift(-HORIZON)` below is the single line that makes this honest. It moves
the realised volatility backwards in time so that each estimate is compared to
what came *after* it. Get that sign wrong and you are grading yourself on the
answer sheet.

In [ ]:
realised = r.rolling(HORIZON).std().shift(-HORIZON) * np.sqrt(BARS_PER_YEAR)

def evaluate(estimate, realised):
    d = pd.concat([estimate.rename('e'), realised.rename('f')], axis=1).dropna()
    err = (d.e - d.f).to_numpy() * 100
    return {'jumpiness': float((d.e * 100).std()),   # how much the estimate moves
            'tracks_future': float(d.e.corr(d.f)),   # how useful it is
            'bias': float(err.mean()),
            'rmse': float(np.sqrt((err ** 2).mean()))}

rows = []
for w in WINDOWS:
    est = r.rolling(w).std() * np.sqrt(BARS_PER_YEAR)
    rows.append({'window': w, 'years': w / BARS_PER_YEAR, **evaluate(est, realised)})
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False, float_format=lambda v: f'{v:14.3f}'))
best = sweep.loc[sweep.rmse.idxmin()]
print(f"\nbest window: {int(best.window)} days ({best.years:.2f} years)")

Read `jumpiness` and `tracks_future` against each other. That contrast is the
whole piece.

Going down the table the estimate gets **steadily smoother**, and **steadily
more useless**. A long window gives you a beautifully stable number that has
almost nothing to do with tomorrow. It is not wrong on average: it describes a
historical average at a moment when you needed to know about now.

In [ ]:
fig, (a, b) = plt.subplots(2, 1, figsize=(7.6, 5.4), sharex=True,
                           gridspec_kw={'height_ratios': [1.35, 1]})
x = np.arange(len(sweep)); k = int(sweep.rmse.idxmin())
a.plot(x, sweep.rmse, marker='o', color='#0e8f5f', lw=2.2)
a.axvline(k, color='#8b98a6', ls=':'); a.set_ylabel('forecast error')
a.set_title('Why there is a best window', loc='left')
b2 = b.twinx()
b.plot(x, sweep.jumpiness, marker='o', color='#2d6fa8')
b2.plot(x, sweep.tracks_future, marker='o', color='#c83c50')
b.set_ylabel('how much it jumps', color='#2d6fa8')
b2.set_ylabel('how well it tracks the future', color='#c83c50'); b2.grid(False)
b.axvline(k, color='#8b98a6', ls=':')
b.set_xticks(x, [str(w) for w in sweep.window], fontsize=8)
b.set_xlabel('days of history'); plt.tight_layout(); plt.show()

## 2. The expected return needs centuries

The standard error of an average return is σ/√T. No model, just arithmetic, and
the result is brutal.

In [ ]:
sigma = r.std() * np.sqrt(BARS_PER_YEAR)
mu    = r.mean() * BARS_PER_YEAR
print(f"volatility {sigma*100:.1f}% a year, average return {mu*100:.2f}%\n")
for yrs in [1, 10, 50, len(r)/BARS_PER_YEAR, 250]:
    se = sigma / np.sqrt(yrs)
    print(f"  {yrs:6.0f} years -> standard error {se*100:5.2f}%"
          f"   95% interval [{(mu-1.96*se)*100:+6.1f}%, {(mu+1.96*se)*100:+6.1f}%]")
print(f"\nFor a standard error of 1 point you would need "
      f"{(sigma/0.01)**2:.0f} YEARS of data.")
print("More frequent sampling does not help: you need more calendar time,")
print("not more points inside the same calendar time (Merton, 1980).")

## 3. Correlation: about a year, and it changed sign

In [ ]:
a_, b_ = PAIR
pa, pb = get_returns(a_, PAIR_START), get_returns(b_, PAIR_START)
both = pd.concat([pa.rename('a'), pb.rename('b')], axis=1).dropna()
fut_corr = both.a.rolling(63).corr(both.b).shift(-63)

rows = []
for w in [21, 63, 126, 252, 504, 1260, 2520]:
    est = both.a.rolling(w).corr(both.b)
    d = pd.concat([est.rename('e'), fut_corr.rename('f')], axis=1).dropna()
    rows.append({'window': w, 'years': w/252,
                 'rmse': float(np.sqrt(((d.e-d.f)**2).mean())),
                 'tracks_future': float(d.e.corr(d.f))})
cor = pd.DataFrame(rows)
print(cor.to_string(index=False, float_format=lambda v: f'{v:14.3f}'))

roll = both.a.rolling(252).corr(both.b)
print(f"\nbest window: {int(cor.loc[cor.rmse.idxmin()].window)} days")
print(f"the one-year correlation went from {roll.min():+.2f} to {roll.max():+.2f}")
print("It CHANGED SIGN. No long average describes that.")

## 4. Rolling, expanding, or exponential?

A flat window has a property nobody defends once it is pointed out: the oldest
day inside it counts exactly as much as yesterday, then tomorrow it counts for
nothing at all. Memory falls off a cliff.

Exponential weighting forgets gradually. `λ` is how much of yesterday's estimate
survives; the half-life is the readable version of the same number.

In [ ]:
rows = []
for lam in LAMBDAS:
    est = r.ewm(alpha=1-lam).std() * np.sqrt(BARS_PER_YEAR)
    rows.append({'lambda': lam, 'half_life_days': np.log(0.5)/np.log(lam),
                 **evaluate(est, realised)})
ew = pd.DataFrame(rows)
print(ew[['lambda','half_life_days','rmse','tracks_future']]
      .to_string(index=False, float_format=lambda v: f'{v:16.3f}'))

exp_est = r.expanding(252).std() * np.sqrt(BARS_PER_YEAR)
best_ew = ew.loc[ew.rmse.idxmin()]
comp = pd.DataFrame([
    {'method': f'rolling {int(best.window)}d', 'rmse': best.rmse, 'tracks': best.tracks_future},
    {'method': 'rolling 252d', 'rmse': float(sweep[sweep.window==252].rmse.iloc[0]),
     'tracks': float(sweep[sweep.window==252].tracks_future.iloc[0])},
    {'method': 'expanding (all history)', **{k: v for k, v in
     zip(['rmse','tracks'], [evaluate(exp_est, realised)['rmse'],
                             evaluate(exp_est, realised)['tracks_future']])}},
    {'method': f"exponential λ={best_ew['lambda']}", 'rmse': best_ew.rmse,
     'tracks': best_ew.tracks_future}])
print('\n' + comp.to_string(index=False, float_format=lambda v: f'{v:12.3f}'))
print(f"\nbest lambda {best_ew['lambda']:.3f}, half-life {best_ew.half_life_days:.0f} days")
print("That is exactly the value J.P. Morgan published in RiskMetrics in 1996.")

**Expanding windows are the worst option available.** Using everything you have
feels like the most rigorous choice and produces the estimate least connected to
the future: it keeps a dead world alive forever.

## 5. And the edge of a strategy?

Take the dip-buying rule from the first piece in this series. Its Sharpe over the
whole sample is one number. Measured one year at a time it is a distribution, and
the distribution is what you would actually have lived through.

In [ ]:
spy = get_returns('SPY', '2000-01-01')

def dip_pnl(r_, lookback=2, holding=1):
    x = r_.to_numpy()
    fired = (pd.Series(x).rolling(lookback).sum() < 0).to_numpy().astype(float)
    fired[:lookback] = 0
    w = pd.Series(fired).rolling(holding).mean().fillna(0).to_numpy()
    pos = np.zeros(len(x)); pos[1:] = w[:-1]
    return pd.Series(pos * x, index=r_.index)

pnl = dip_pnl(spy)
full = pnl.mean() / pnl.std() * np.sqrt(252)
print(f"Sharpe over the whole sample: {full:.2f}\n")

rows = []
for w in [126, 252, 504, 1260]:
    rs = (pnl.rolling(w).mean() / pnl.rolling(w).std() * np.sqrt(252)).dropna()
    rows.append({'window': w, 'years': w/252, 'worst': rs.min(), 'median': rs.median(),
                 'best': rs.max(), 'pct negative': (rs < 0).mean()*100})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:13.2f}'))

rs = (pnl.rolling(252).mean() / pnl.rolling(252).std() * np.sqrt(252)).dropna()
fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.fill_between(rs.index, 0, rs.values, where=rs.values >= 0, color='#0e8f5f', alpha=.45)
ax.fill_between(rs.index, 0, rs.values, where=rs.values < 0, color='#c83c50', alpha=.45)
ax.axhline(full, color='#2d6fa8', ls='--', lw=1.8, label=f'the published number: {full:.2f}')
ax.axhline(0, color='k', lw=1); ax.legend()
ax.set_ylabel('Sharpe over the trailing year'); plt.show()

## 6. Things worth trying yourself

1. **Change `HORIZON` to 5, then to 252.** The optimal window should move with
   it. If it does, you have shown that the answer belongs to the question and
   not to volatility.
2. **Set `SYMBOL = 'BTC-USD'`.** A younger, wilder market. Does the optimum stay
   at three weeks, or does the higher noise push it longer?
3. **Change `PAIR` to `('SPY','GLD')`.** Gold against stocks did not flip sign
   the way bonds did. Does the optimal correlation window lengthen?
4. **Add a GARCH(1,1) forecast** and put it in the comparison table. It should
   beat the exponential, and the gap tells you what the extra complexity buys.

Found something I got wrong? There is a form at the bottom of the article.